# Introduction

The goal is to verify the alignment of the sample peaks in a batch and make sure batch aggregated peaks are correct.

# Database Initialization

In [ ]:
from mascope_backend.db import init_db

await init_db()

# Data Collection

The pipeline for data collection is taken from `get_sample_batch_peaks` controller. However, the pipeline will be simplified here for clarity:
- intensity_variable is hardcoded
- only one ionization mode is picked (the most frequently occurring one)

In [ ]:
from collections import Counter
import asyncio
import numpy as np
import mascope_signal.compute as m_compute
import mascope_file.io as m_io
from mascope_tools.alignment.calibration import CentroidedSpectrum, Spectra

from sqlalchemy import select
from mascope_backend.api.controllers.sample.batches.sample_batches_controller import get_sample_batch
from mascope_backend.api.new.instrument_configs.lib import (
    read_instrument_functions,
)
from mascope_backend.db import async_session
from mascope_backend.db.models import Sample


sample_batch_id = "EHcWyasdCruYN9JT"
intensity_variable = "sum_peak_heights"

# Load batch data
batch_response = await get_sample_batch(sample_batch_id)
sample_batch = batch_response.get("data")

# Get samples in the batch
async with async_session() as session:
    stmt = select(Sample).where(Sample.sample_batch_id == sample_batch_id)
    result = await session.execute(stmt)
    sample_items = result.scalars().all()

# Load instrument functions
resolution_functions = dict()
for item in sample_items:
    _, resolution_func = await read_instrument_functions(item.filename)
    resolution_functions[item.filename] = resolution_func

# Pick the most frequently occurring ionization mode
ionization_modes = [item.ionization_mode_id for item in sample_items]
ionization_mode_counts = Counter(ionization_modes)
ionization_mode_pick = ionization_mode_counts.most_common(1)[0][0]

semaphore = asyncio.Semaphore(6)

def _sync_load_peak_data(filename, polarity):
    """Synchronous helper to load peak data in a thread."""
    timestamps = m_compute.get_scan_timestamps(filename, polarity=polarity)

    peak_data = m_io.load_peak_data(filename)
    peak_id = peak_data["peak_id"].values
    polarity_coord = peak_data["polarity"].values
    peak_data = peak_data[intensity_variable]
    mz_mask = polarity_coord == polarity
    mz = peak_data["mz"].values[mz_mask]
    intensity = peak_data.values[mz_mask] / timestamps.size
    peak_id = peak_id[mz_mask]

    return mz, intensity, peak_id

async def _prepare_spec(sample_item):
    """Prepare CentroidedSpectrum for a sample item."""
    filename = sample_item.filename
    polarity = sample_item.polarity
    ionization_mode = sample_item.ionization_mode_id

    if ionization_mode != ionization_mode_pick:
        # Skip loading data for non-picked ionization modes
        return ionization_mode, None

    async with semaphore:
        mz, intensity, peak_id = await asyncio.to_thread(
            _sync_load_peak_data, filename, polarity
        )
    resolution = resolution_functions[filename](mz)
    signal_to_noise = np.ones(mz.size)

    # Placeholder S/N values since they are not required for alignment
    spec = CentroidedSpectrum(
        mz=mz,
        intensity=intensity,
        resolution=resolution,
        signal_to_noise=signal_to_noise,
        peak_id=peak_id,
    )
    return ionization_mode, spec

# Collect spectra concurrently
collected_specs = await asyncio.gather(
    *[_prepare_spec(item) for item in sample_items]
)

sample_peak_list = list()
for ion_mode, spec in collected_specs:
    if ion_mode == ionization_mode_pick:
        sample_peak_list.append(spec)

# Align and sum peaks
vlm_min_mzs, vlm_max_mzs = set(), set()
peak_collection = Spectra(sample_peak_list, timestamps=np.arange(len(sample_peak_list)))

batch_peaks, vlm_min_mz, vlm_max_mz = m_compute.sum_peak_collection(
    peak_collection
)

# Visualization

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

current_idx = 0
all_mzs = np.concatenate([spec.mz for spec in sample_peak_list])
all_intensities = np.concatenate([spec.intensity for spec in sample_peak_list])

def plot_peaks(idx):
    clear_output(wait=True)
    plt.figure(figsize=(10, 6))
    mz = batch_peaks.mz[idx]
    fwhm = mz / batch_peaks.resolution[idx]
    x_range = mz-fwhm, mz+fwhm
    plt.axvline(x=mz, color="#882020", linestyle='-', linewidth=3, label='Batch Peak')
    plt.axvspan(mz - fwhm / 2, mz + fwhm / 2, color='orange', alpha=0.2, label='FWHM Band')
    
    mz_mask = (all_mzs >= x_range[0]) & (all_mzs <= x_range[1])
    plt.scatter(x=all_mzs[mz_mask], y=all_intensities[mz_mask], s=100, alpha=0.5, label='Non-Aligned Sample Peaks')
    plt.title(f'Batch Peak {idx+1}/{len(batch_peaks.mz)}')
    plt.xlabel('m/z')
    plt.ylabel('Intensity')
    plt.xlim(x_range)
    plt.legend()
    plt.show()
    display(widgets.HBox([prev_button, next_button]))

def on_next(b):
    global current_idx
    if current_idx < len(batch_peaks.mz) - 1:
        current_idx += 1
    plot_peaks(current_idx)

def on_prev(b):
    global current_idx
    if current_idx > 0:
        current_idx -= 1
    plot_peaks(current_idx)

prev_button = widgets.Button(description="Previous")
next_button = widgets.Button(description="Next")
prev_button.on_click(on_prev)
next_button.on_click(on_next)

plot_peaks(current_idx)